# 🇧🇷 EDA — Indicadores Macroeconômicos (2000–2026)

5 séries: SELIC, IPCA, câmbio USD/BRL, IGP-M e IBC-Br.

**Princípios desta versão** (correções de review econométrico):
1. **Meta de inflação por ano** — a banda mudou ao longo do tempo
2. **Corte comum** — meses sem o alvo fechado não são analisados
3. **Variações, não níveis** — correlação de série não estacionária é espúria
4. **Estacionaridade testada** (ADF) antes de qualquer correlação
5. **Lags com fonte única** — o scatter usa o pico da própria tabela

| Bloco | Pergunta |
|---|---|
| 0 | Os dados são confiáveis? |
| 1 | As séries são estacionárias? |
| 2 | Como se distribuem as variações? |
| 3 | Os eventos históricos batem com as séries? |
| 4 | A meta foi cumprida, ano a ano? |
| 5 | Existe sazonalidade no IPCA? |
| 6 | A volatilidade é persistente? |
| 7 | Quem antecede a inflação — em VARIAÇÕES? ⭐ |
| 8 | Pass-through cambial e ciclo de juros |
| 9 | Conclusões → features |


## Setup e corte comum

**Pergunta:** sobre quais dados estamos analisando?

**Decisão:** a amostra analítica termina no último mês com IPCA fechado —
meses posteriores têm séries desalinhadas (IBC-Br atrasa ~2 meses, IGP-M
fecha dia 30) e gerariam NaN em cascata nos cálculos.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd()
while not (RAIZ / 'src').is_dir() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from src.database.db import get_engine

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

load_dotenv(RAIZ / '.env')
engine = get_engine()

df = pd.read_sql('SELECT * FROM staging.indicadores_macro ORDER BY data', engine)
df['data'] = pd.to_datetime(df['data'])
df = df.set_index('data')

# CORTE COMUM: último mês com IPCA fechado = fim da amostra analítica
CORTE = df['ipca_pct'].last_valid_index()
d = df.loc[:CORTE].copy()

print(f'Banco: {df.shape[0]} meses | Análise: {d.shape[0]} meses (até {CORTE:%b/%Y})')
d.tail()

## 0. Auditoria de qualidade

**Pergunta:** os nulos são estruturais (publicação atrasada) ou falha de extração?

**O que observar:** nulos no FIM de cada série = atraso de publicação (normal).
Nulos no MEIO = suspeita. IBC-Br começa em 2003 (série nasce lá — ver config `inicio`).

In [ ]:
resumo = pd.DataFrame({
    'n_nulos': d.isna().sum(),
    'primeiro_dado': d.apply(lambda s: s.first_valid_index()),
    'ultimo_dado': d.apply(lambda s: s.last_valid_index()),
})
resumo['primeiro_dado'] = pd.to_datetime(resumo['primeiro_dado'])
resumo['ultimo_dado'] = pd.to_datetime(resumo['ultimo_dado'])

ultima = d.index.max().to_period('M')
ultimo_periodo = resumo['ultimo_dado'].dt.to_period('M')

resumo['atraso_meses'] = (
    (ultima.year - ultimo_periodo.dt.year) * 12
    + (ultima.month - ultimo_periodo.dt.month)
).astype('int64')
resumo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(d.isna().T, cmap=sns.color_palette(['#2166AC', '#D73027'], as_cmap=True),
            cbar=False, ax=ax)
ax.set_title(f'Mapa de nulos até {CORTE:%b/%Y} (azul = preenchido, vermelho = sem dado)')
plt.tight_layout(); plt.show()

## 1. Estacionaridade — teste antes de correlacionar

**Pergunta:** as séries são estacionárias?

**Por quê:** correlacionar NÍVEIS de séries com tendência (câmbio 2→6 reais,
IBC-Br 100→150) mede tendência comum, não relação econômica. O correto é
trabalhar com **variações**. ADF (Augmented Dickey-Fuller): p < 0,05 = estacionária.

**Conclusão (preencha após rodar):** _anote quais colunas de `var` precisam
de mais diferenciação — isso orienta a ordem do ARIMA na Etapa ML._

In [ ]:
from statsmodels.tsa.stattools import adfuller

# VARIAÇÕES — linguagem econômica: choque hoje → resposta depois
var = pd.DataFrame({
    'ipca':          d['ipca_pct'],
    'd_cambio':      d['cambio_usd'].pct_change() * 100,
    'd_selic':       d['taxa_selic'].diff(),
    'd_ibcbr':       d['ibcbr'].pct_change() * 100,
    'd_ipca_focus':  d['ipca_focus'].diff(),      # ⭐ revisão da expectativa (em p.p.)
    'desemprego': d['desemprego'], 
})

print('Teste ADF (H0: não estacionária):')
for col in var.columns:
    p = adfuller(var[col].dropna())[1]
    print(f'  {col:10s} p = {p:.4f}  {"estacionária" if p < 0.05 else "NÃO estacionária"}')

var.describe().T.round(2)

## 2. Distribuições das variações

**Pergunta:** as variações têm caudas pesadas? Outliers batem com crises?

**O que observar:** `d_cambio` com cauda à direita (desvalorizações são
bruscas); `d_ibcbr` concentrado perto de zero com outliers em 2008/2020;
histograma de NÍVEL do IBC-Br seria só uma rampa — por isso variações.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, var.columns):
    sns.histplot(var[col], kde=True, ax=ax, bins=40)
    ax.set_title(col)
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

## 3. Séries no tempo + eventos históricos

**Pergunta:** os eventos batem com o movimento das séries?

**Como usar:** a lista de eventos é hipótese sua — edite livremente. O que o
olho sugerir aqui, o bloco 7 confere numericamente depois.

In [ ]:
eventos = {
    '2002-10': 'Eleição Lula',
    '2008-09': 'Crise global',
    '2015-01': 'Crise econômica',
    '2016-08': 'Impeachment',
    '2020-03': 'Pandemia',
    '2021-02': 'Vacinação',
    '2022-03': 'Guerra Ucrânia',
    '2023-01': 'Novo governo',
}
eventos = {pd.to_datetime(k): v for k, v in eventos.items()}

fig, axes = plt.subplots(len(d.columns), 1,
                         figsize=(18, 4 * len(d.columns)), sharex=True)
if len(d.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, d.columns):
    ax.plot(d.index, d[col], color='#2166AC', lw=1.8)
    if col in ('ipca_pct', 'igpm'):
        ax.axhline(0, color='black', lw=1, ls='--', alpha=0.8)
    for i, (data, nome) in enumerate(eventos.items()):
        if data < d.index.min() or data > d.index.max():
            continue
        cor = '#D73027' if i % 2 == 0 else '#762A83'
        ax.axvspan(data - pd.Timedelta(days=20), data + pd.Timedelta(days=20),
                   color=cor, alpha=0.10, zorder=1)
        ax.axvline(data, color=cor, lw=2, ls='--', alpha=0.9, zorder=4)
        ax.text(data, 0.97, nome, transform=ax.get_xaxis_transform(),
                rotation=90, ha='right', va='top', fontsize=8, fontweight='bold',
                color=cor,
                bbox=dict(facecolor='white', edgecolor=cor, alpha=0.9,
                          boxstyle='round,pad=0.3'),
                zorder=5)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.25)

axes[-1].set_xlabel('Data')
fig.suptitle('Indicadores econômicos e principais eventos históricos',
             fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

## 4. IPCA acumulado 12m vs meta **vigente em cada ano**

**Pergunta:** a meta foi cumprida, considerando a meta E a tolerância do ano?

**⚠️ Correção importante:** a meta brasileira MUDOU ao longo do tempo
(6% em 2000, 4,5% de 2005–2018, queda gradual até 3% em 2024+; tolerância
±2 p.p. até 2006, ±1,5 p.p. depois). Aplicar a banda atual em 2000 distorce
qualquer conclusão. Fonte dos valores: Banco Central — confira em
bcb.gov.br/controleinflacao antes de publicar.

**Conclusão (preencha):** _% dentro da banda vigente; períodos mais longos
fora da banda; tempo médio de retorno._

In [ ]:
# --------------------------------------------------------------
# 1. Metas de inflação e tolerâncias históricas
# Fonte: Banco Central do Brasil / CMN
# Valores em percentual
# --------------------------------------------------------------

metas_bc = pd.DataFrame({
    "ano": list(range(2000, 2027)),

    "meta": [
        6.00,  # 2000
        4.00,  # 2001
        3.50,  # 2002
        4.00,  # 2003
        5.50,  # 2004
        5.50,  # 2005
        4.50,  # 2006
        4.50,  # 2007
        4.50,  # 2008
        4.50,  # 2009
        4.50,  # 2010
        4.50,  # 2011
        4.50,  # 2012
        4.50,  # 2013
        4.50,  # 2014
        4.50,  # 2015
        4.50,  # 2016
        4.50,  # 2017
        4.50,  # 2018
        4.25,  # 2019
        4.00,  # 2020
        3.75,  # 2021
        3.50,  # 2022
        3.25,  # 2023
        3.00,  # 2024
        3.00,  # 2025
        3.00,  # 2026
    ],

    "tolerancia": [
        2.00,  # 2000
        2.00,  # 2001
        2.00,  # 2002
        2.50,  # 2003
        2.50,  # 2004
        2.50,  # 2005
        2.00,  # 2006
        2.00,  # 2007
        2.00,  # 2008
        2.00,  # 2009
        2.00,  # 2010
        2.00,  # 2011
        2.00,  # 2012
        2.00,  # 2013
        2.00,  # 2014
        2.00,  # 2015
        2.00,  # 2016
        1.50,  # 2017
        1.50,  # 2018
        1.50,  # 2019
        1.50,  # 2020
        1.50,  # 2021
        1.50,  # 2022
        1.50,  # 2023
        1.50,  # 2024
        1.50,  # 2025
        1.50,  # 2026
    ],
})

# --------------------------------------------------------------
# 2. Prepara a série do IPCA
# --------------------------------------------------------------

d = df.copy()

# Garante índice temporal
d.index = pd.to_datetime(d.index)
d = d.sort_index()

if "ipca_pct" not in d.columns:
    raise KeyError("A coluna 'ipca_pct' não foi encontrada no DataFrame.")

# Remove valores ausentes
ipca_mensal = d["ipca_pct"].astype(float)

# --------------------------------------------------------------
# 3. Calcula o IPCA acumulado em 12 meses
# --------------------------------------------------------------

# Para IPCA mensal em porcentagem:
# 0.50 significa 0,50% no mês
ipca12 = (
    (1 + ipca_mensal / 100)
    .rolling(window=12, min_periods=12)
    .apply(np.prod, raw=True)
    .sub(1)
    .mul(100)
    .rename("ipca12")
)

df_m = ipca12.to_frame().reset_index()

# Identifica corretamente o nome da coluna de data
coluna_data = df_m.columns[0]
df_m = df_m.rename(columns={coluna_data: "data"})

df_m["ano"] = df_m["data"].dt.year

# --------------------------------------------------------------
# 4. Junta o IPCA com a meta correspondente a cada ano
# --------------------------------------------------------------

df_m = df_m.merge(
    metas_bc,
    on="ano",
    how="left",
    validate="many_to_one"
)

# Verifica se existem anos sem meta cadastrada
anos_sem_meta = (
    df_m.loc[df_m["meta"].isna(), "ano"]
    .dropna()
    .unique()
)

if len(anos_sem_meta) > 0:
    raise ValueError(
        f"Não há meta cadastrada para os anos: {anos_sem_meta.tolist()}"
    )

# Limites da banda
df_m["limite_inferior"] = (
    df_m["meta"] - df_m["tolerancia"]
)

df_m["limite_superior"] = (
    df_m["meta"] + df_m["tolerancia"]
)

# Classificação do IPCA em relação à banda
df_m["situacao"] = np.select(
    [
        df_m["ipca12"] < df_m["limite_inferior"],
        df_m["ipca12"] > df_m["limite_superior"],
    ],
    [
        "Abaixo da banda",
        "Acima da banda",
    ],
    default="Dentro da banda"
)

# --------------------------------------------------------------
# 5. Gráfico
# --------------------------------------------------------------

fig, ax = plt.subplots(figsize=(17, 7))

# Banda de tolerância variável ao longo do tempo
ax.fill_between(
    df_m["data"],
    df_m["limite_inferior"],
    df_m["limite_superior"],
    color="#74ADD1",
    alpha=0.28,
    label="Banda de tolerância"
)

# Limite inferior
ax.plot(
    df_m["data"],
    df_m["limite_inferior"],
    color="#2166AC",
    linestyle="--",
    linewidth=1.3,
    label="Limite inferior"
)

# Limite superior
ax.plot(
    df_m["data"],
    df_m["limite_superior"],
    color="#D73027",
    linestyle="--",
    linewidth=1.3,
    label="Limite superior"
)

# Meta central variável
ax.step(
    df_m["data"],
    df_m["meta"],
    where="post",
    color="black",
    linestyle=":",
    linewidth=2,
    label="Meta do BC"
)

# IPCA acumulado em 12 meses
ax.plot(
    df_m["data"],
    df_m["ipca12"],
    color="#E67E22",
    linewidth=2.5,
    label="IPCA acumulado em 12 meses",
    zorder=4
)

# Pontos fora da banda
fora_da_banda = (
    (df_m["ipca12"] < df_m["limite_inferior"]) |
    (df_m["ipca12"] > df_m["limite_superior"])
)

ax.scatter(
    df_m.loc[fora_da_banda, "data"],
    df_m.loc[fora_da_banda, "ipca12"],
    color="#C0392B",
    edgecolor="white",
    linewidth=0.7,
    s=42,
    zorder=5,
    label="Fora da banda"
)

# Linhas verticais indicando mudança anual da meta
anos_plotados = sorted(df_m["ano"].dropna().unique())

for ano in anos_plotados:
    if ano == anos_plotados[0]:
        continue

    data_inicio_ano = pd.Timestamp(f"{int(ano)}-01-01")

    ax.axvline(
        data_inicio_ano,
        color="gray",
        linestyle=":",
        linewidth=0.7,
        alpha=0.35
    )

# Formatação
ax.set_title(
    "IPCA acumulado em 12 meses versus metas do Banco Central",
    fontsize=16,
    fontweight="bold"
)

ax.set_xlabel("Data")
ax.set_ylabel("Inflação acumulada em 12 meses (%)")
ax.grid(True, linestyle=":", alpha=0.35)
ax.legend(loc="upper left", ncol=2)

plt.tight_layout()
plt.show()

# --------------------------------------------------------------
# 6. Resumo
# --------------------------------------------------------------

resumo = pd.DataFrame({
    "observações válidas": [df_m["ipca12"].notna().sum()],
    "meses dentro da banda": [
        (df_m["situacao"] == "Dentro da banda").sum()
    ],
    "meses acima da banda": [
        (df_m["situacao"] == "Acima da banda").sum()
    ],
    "meses abaixo da banda": [
        (df_m["situacao"] == "Abaixo da banda").sum()
    ],
})

display(resumo)

# Mostra a tabela anual de metas utilizadas
display(
    metas_bc
    .set_index("ano")
    .assign(
        limite_inferior=lambda x: x["meta"] - x["tolerancia"],
        limite_superior=lambda x: x["meta"] + x["tolerancia"]
    )
    .round(2)
)

## 5. Sazonalidade do IPCA

**Pergunta:** alguns meses são sistematicamente mais inflacionários?

**Correção:** média simples é dominada por anos extremos (2002: +3% em
novembro). Usamos **mediana por mês** + **anomalia** (IPCA do mês menos a
média do próprio ano), que remove o efeito do nível geral de inflação.

In [ ]:
ipca_mes = d['ipca_pct'].to_frame('ipca')
ipca_mes['mes'] = ipca_mes.index.month
ipca_mes['media_ano'] = ipca_mes.groupby(ipca_mes.index.year)['ipca'].transform('mean')
ipca_mes['anomalia'] = ipca_mes['ipca'] - ipca_mes['media_ano']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=ipca_mes, x='mes', y='anomalia', ax=axes[0])
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Anomalia de IPCA por mês (mês − média do ano)')

mediana_mes = ipca_mes.groupby('mes')['anomalia'].median()
mediana_mes.plot.bar(ax=axes[1], color='#c0392b', alpha=0.8)
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Mediana da anomalia por mês')

plt.tight_layout(); plt.show()
mediana_mes.round(2).sort_values(ascending=False)

## 6. Volatilidade (desvio padrão móvel 12m)

**Pergunta:** a volatilidade é persistente (clusters)?

**O que observar:** vol alta seguindo vol alta = heteroscedasticidade —
relevante pra modelagem de variância (GARCH) no futuro, e justifica
medidas de robustez no baseline ML.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, col in zip(axes.flat, ['ipca_pct', 'igpm', 'cambio_usd', 'taxa_selic', 'd_ipca_focus', 'desemprego']):
    ax.plot(d[col].rolling(12).std(), lw=1.2)
    ax.set_title(f'Volatilidade 12m — {col}')
plt.tight_layout(); plt.show()

## 7. Quem antecede a inflação? Correlação por lags **em variações** ⭐

**Pergunta:** qual variação hoje antecede o IPCA de daqui a quantos meses?

**Correções desta versão:**
- Correlação entre VARIAÇÕES (nível é espúrio — bloco 1)
- IGP-M fica de fora das features de previsão: ele no lag 0 é concorrência
  (inflação contra inflação), um **nowcast**, não um preditor
- A tabela de picos abaixo ALIMENTA o bloco 8 — um único lugar define os lags

**Conclusão (preencha):** _copie a tabela 'Lag de pico' pro bloco 9._
Referência da literatura: pass-through cambial 2–6m, IGP-M 1–3m.

In [ ]:
LAGS = range(0, 13)
PREDITORES = ['d_cambio', 'd_selic', 'd_ibcbr', 'd_ipca_focus', 'desemprego']

corr_lag = pd.DataFrame({
    col: [var[col].shift(l).corr(var['ipca']) for l in LAGS]
    for col in PREDITORES
}, index=LAGS)

fig, ax = plt.subplots(figsize=(12, 5))
for col in corr_lag.columns:
    ax.plot(corr_lag.index, corr_lag[col], marker='o', label=col)
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('defasagem (meses)'); ax.set_ylabel('correlação com IPCA')
ax.set_title('Correlação (em variações) com IPCA por defasagem')
ax.legend(); plt.tight_layout(); plt.show()

# FONTE ÚNICA: os lags de pico saem daqui e alimentam o bloco 8
picos = corr_lag.abs().idxmax()
tabela_picos = pd.DataFrame({
    'lag': picos,
    'correlação': [
        corr_lag.iloc[int(picos[c])][c]
        for c in picos.index
    ]
}).round(2)
print('Lag de maior |correlação| com IPCA (em variações):')
print(tabela_picos)

## 8. Pass-through cambial e ciclo de juros — nos lags da própria análise

**Pergunta:** a relação existe nos lags de pico — e ela muda de regime?

**Correção:** os lags abaixo vêm da tabela do bloco 7 (`picos`) — não há
número mágico solto. Ajuste `LAG_CAMBIO`/`LAG_SELIC` manualmente só para
explorar, sabendo que o valor "oficial" é o do bloco 7.

**O que observar:** nuvem ascendente em câmbio = pass-through; SELIC com
correlação positiva é esperado no curto prazo (o BC **reage** à inflação —
causalidade inversa); cores por ano revelam regimes diferentes.

In [ ]:
# lags oficiais = picos da tabela (fonte única)
LAG_CAMBIO = int(picos['d_cambio'])
LAG_SELIC  = int(picos['d_selic'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

x = var['d_cambio']; y = var['ipca'].shift(-LAG_CAMBIO)
sc = axes[0].scatter(x, y, c=var.index.year, cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Δ% câmbio no mês t')
axes[0].set_ylabel(f'IPCA em t+{LAG_CAMBIO}')
axes[0].set_title(f'Pass-through: Δcâmbio vs IPCA {LAG_CAMBIO}m depois (r={x.corr(y):.2f})')
plt.colorbar(sc, ax=axes[0], label='ano')

x2 = var['d_selic']; y2 = var['ipca'].shift(-LAG_SELIC)
sc2 = axes[1].scatter(x2, y2, c=var.index.year, cmap='plasma', alpha=0.6)
axes[1].set_xlabel('Δ Selic (p.p.) no mês t')
axes[1].set_ylabel(f'IPCA em t+{LAG_SELIC}')
axes[1].set_title(f'ΔSelic vs IPCA {LAG_SELIC}m depois (r={x2.corr(y2):.2f})')
plt.colorbar(sc2, ax=axes[1], label='ano')

plt.tight_layout(); plt.show()

## 9. Conclusões → features pro ML

**Preencha com o que você observou** (valores reais do bloco 7):

```
Série      | Lag de pico | Correlação | Feature pro modelo
-----------|-------------|------------|----------------------
d_cambio   |      4      |    0.26    | cambio_usd.pct_change().shift(__)
d_selic    |      1      |    0.38    | taxa_selic.diff().shift(__)
d_ibcbr    |      1      |    0.12    | ibcbr.pct_change().shift(__)
sazonalid. | meses: 12   |    —       | mes (categórica)
```

### Checklist Etapa ML
- [ ] Lags copiados do bloco 7 (não do chute)
- [ ] Colunas não estacionárias anotadas no bloco 1 (orienta ordem do ARIMA)
- [ ] IGP-M excluído das features de previsão (é nowcast — lag 0)
- [ ] Tratamento dos nulos de 2003 (início do IBC-Br) definido
- [ ] Baselines: naive + média móvel 12m + ARIMA
- [ ] Validação: rolling origin, NUNCA shuffle

### Pendências registradas (EDA v4)
- [ ] Correlação móvel + testes de quebra estrutural (2003 / 2015 / pandemia)
- [ ] Granger / ACF-PACF / cross-correlação com pré-branqueamento
- [ ] Expectativas do Boletim Focus (melhor preditor de IPCA) — vira linha no SERIES
